# Customer Lifetime Value (CLV) Analytics Platform

**End-to-end pipeline:** Data Generation > Feature Engineering > EDA > Segmentation > Model Training > Explainability

| Author | Role |
|--------|------|
| **Sanman** | Lead Developer |
| **Varsha** | Co-Developer |

---

## 1. Setup and Imports

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

warnings.filterwarnings('ignore')
%matplotlib inline

# Add project root to path
ROOT = os.path.abspath('..')
sys.path.insert(0, ROOT)

from src.data_generator import generate_and_save
from src.feature_engineering import build_feature_matrix, get_feature_columns
from src.segmentation import segment_customers
from src.models import train_and_evaluate
from src.explainability import explain_model

print('All imports successful.')

## 2. Data Generation

Generate 50,000 synthetic customers and 350K+ transactions with realistic patterns:
- Pareto-distributed spending (whale behavior)
- Seasonal purchase patterns (Q4 peaks)
- Age-correlated category preferences
- Churn simulation (~30% inactive)

In [ ]:
data_dir = os.path.join(ROOT, 'data', 'raw')
customers, transactions = generate_and_save(data_dir)

print(f'Customers: {customers.shape}')
print(f'Transactions: {transactions.shape}')
customers.head()

In [ ]:
transactions.head(10)

## 3. Exploratory Data Analysis

In [ ]:
print('--- Customer Demographics ---')
print(f'Age range: {customers["age"].min()} - {customers["age"].max()}')
print(f'Gender distribution:\n{customers["gender"].value_counts().to_string()}')
print(f'\nAcquisition channels:\n{customers["acquisition_channel"].value_counts().to_string()}')

In [ ]:
# Revenue distribution per customer
rev_per_cust = transactions.groupby('customer_id')['amount'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(rev_per_cust, bins=80, color='#6C63FF', alpha=0.85, edgecolor='none')
axes[0].set_yscale('log')
axes[0].set_title('Revenue Distribution (Log Scale)')
axes[0].set_xlabel('Total Revenue per Customer ($)')
axes[0].axvline(rev_per_cust.median(), color='#FF6584', linestyle='--', label=f'Median: ${rev_per_cust.median():,.0f}')
axes[0].legend()

axes[1].boxplot(rev_per_cust, vert=True, patch_artist=True,
                boxprops=dict(facecolor='#6C63FF', alpha=0.7),
                medianprops=dict(color='#FF6584', linewidth=2))
axes[1].set_title('Revenue Box Plot')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly revenue trend
monthly = transactions.set_index('date').resample('M')['amount'].sum().reset_index()
monthly.columns = ['month', 'revenue']

fig = px.area(monthly, x='month', y='revenue', title='Monthly Revenue Trend')
fig.show()

In [ ]:
# Category revenue breakdown
cat_rev = transactions.groupby('product_category')['amount'].sum().sort_values(ascending=True)
fig = px.bar(cat_rev.reset_index(), x='amount', y='product_category', orientation='h',
             title='Revenue by Product Category', color_discrete_sequence=['#43E97B'])
fig.show()

## 4. Feature Engineering (Leakage-Free)

**Critical design decision:** We split the dataset temporally into a calibration period and a 6-month holdout period.
- Features (RFM, behavioral) are computed ONLY on the calibration period.
- The target variable (CLV) is defined as revenue in the holdout period.
- This prevents data leakage and ensures realistic evaluation.

In [ ]:
customers = pd.read_csv(os.path.join(data_dir, 'customers.csv'), parse_dates=['signup_date'])
transactions = pd.read_csv(os.path.join(data_dir, 'transactions.csv'), parse_dates=['date'])

features = build_feature_matrix(customers, transactions)
print(f'\nFeature matrix shape: {features.shape}')
features.head()

In [ ]:
feature_cols = get_feature_columns(features)
print(f'Number of features: {len(feature_cols)}')
print(f'Features: {feature_cols}')

In [ ]:
# Correlation heatmap of key features
key_feats = ['recency', 'frequency', 'monetary', 'avg_order_value', 'category_diversity',
             'tenure_days', 'purchase_velocity', 'clv']
key_feats = [f for f in key_feats if f in features.columns]
corr = features[key_feats].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Customer Segmentation (K-Means)

In [ ]:
reports_dir = os.path.join(ROOT, 'reports')
features = segment_customers(features, os.path.join(reports_dir, 'figures'))

# Segment summary
seg_summary = features.groupby('segment').agg(
    Customers=('customer_id', 'count'),
    Avg_CLV=('clv', 'mean'),
    Avg_Recency=('recency', 'mean'),
    Avg_Frequency=('frequency', 'mean'),
).round(1)
seg_summary

In [ ]:
fig = px.pie(features, names='segment', title='Customer Segment Distribution',
             color_discrete_sequence=['#6C63FF', '#FF6584', '#43E97B', '#FFD93D'])
fig.update_traces(hole=0.4)
fig.show()

## 6. Model Training and Evaluation

We train and compare 5 models:
1. **Ridge Regression** (baseline)
2. **Random Forest** (intermediate)
3. **XGBoost** (Optuna-tuned, 30 trials)
4. **LightGBM** (Optuna-tuned, 30 trials)
5. **BG/NBD + Gamma-Gamma** (probabilistic / BTYD)

In [ ]:
feature_cols = get_feature_columns(features)
models_dir = os.path.join(ROOT, 'models')

models, results_df, best_model, best_key, splits = train_and_evaluate(
    features, feature_cols, models_dir=models_dir, reports_dir=reports_dir, n_tune_trials=30
)

print('\n--- Model Comparison ---')
results_df

In [ ]:
# Visual comparison
fig = px.bar(results_df, x='Model', y='R2', color='Model',
             title='R-squared Score Comparison',
             color_discrete_sequence=['#6C63FF', '#FF6584', '#43E97B', '#FFD93D', '#00C9FF'])
fig.show()

In [ ]:
# Residual analysis for best model
X_train, X_test, y_train, y_test = splits
preds = best_model.predict(X_test)
residuals = y_test - preds

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(preds, residuals, alpha=0.1, s=5, color='#6C63FF')
axes[0].axhline(0, color='#FF6584', linewidth=2, linestyle='--')
axes[0].set_title(f'Residuals - {best_key}')
axes[0].set_xlabel('Predicted CLV')
axes[0].set_ylabel('Residual')

axes[1].hist(residuals, bins=60, color='#6C63FF', edgecolor='none', alpha=0.8)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.show()

## 7. SHAP Explainability

In [ ]:
shap_values = explain_model(
    best_model, X_train, X_test, feature_cols,
    os.path.join(reports_dir, 'figures'), model_type='tree'
)

## 8. Key Business Insights

| Insight | Recommendation |
|---------|----------------|
| Champions segment drives majority of future revenue | Prioritize retention budgets for this tier |
| Customers inactive for 90+ days rarely return | Deploy automated win-back campaigns at day 60 |
| Referral channel yields highest-value customers | Increase referral incentive programs |
| Customers buying from 3+ categories have higher CLV | Implement cross-sell strategies |
| Q4 (Nov-Dec) drives ~35% of annual revenue | Plan inventory and campaigns accordingly |

---

## 9. Conclusion

This pipeline demonstrates a production-ready approach to CLV prediction:
- **Leakage-free** temporal feature engineering
- **Multi-model** comparison (ML + Probabilistic)
- **Actionable** customer segmentation
- **Interpretable** SHAP explanations

Launch the interactive dashboard with:
```bash
streamlit run app/dashboard.py
```